# 9. Candidate Selection by LLM + Library Expansion (MMPA)

## 이번 노트북에서 할 것
- 치환 후보(candidates) 선택도 LLM이 판단하도록 확장
  (현재는 candidate_idx=0으로 고정 — 여러 후보 중 상황에 맞는 것 선택하기)
- rdMMPA를 활용해 Tox21 데이터에서 자동으로 치환쌍(matched molecular pair) 추출
  → replacement_library를 데이터 기반으로 자동 확장하는 실험

## 간략한 정리 (08까지)
- 도구 계층 + 반복 루프 + "어떤 문제부터 고칠지" LLM 판단 계층까지 완성
- agent.py: ask_llm_which_problem_to_fix() — Gemini 3.5 Flash, JSON 강제 출력,
  파싱 실패시 규칙기반 fallback
- 비교실험 완료: 같은 분자에서 규칙기반(aniline 먼저)과 LLM기반(nitro_group 먼저,
  화학적 근거 제시)이 다른 경로로 각각 success — LLM 판단력을 실증
- molecule_editor.py에 iterative_fix_loop(llm_client=None/client)로 두 모드 다 지원

## 다음에 해야 할 것 (오늘 끝나면)
- candidate 선택까지 LLM화되면, 진짜 "완전한 에이전트 루프" 1차 완성
- Held-out set 전체에 루프 적용 → 성공률/개선율 통계 (평가지표 확정)
- 실제 사례(개발중단 약물 등) 케이스스터디 1~2개 준비
- 제안서(hwpx) 작성 시작 (마감 8/7 — 남은 시간 고려해 이제부터는 문서화도 병행)

In [1]:
# 셀 1
!pip install rdkit -q
!pip install -U google-genai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 48.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 28.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.2 which is incompatible.


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 99 (delta 36), reused 66 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 272.53 KiB | 1.10 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.agent import ask_llm_which_problem_to_fix

data = load_tox21_clean()
print("도구 로드 확인 완료")

[04:38:12] WARNING: not removing hydrogen atom without neighbors
[04:38:12] Explicit valence for atom # 8 Al, 6, is greater than permitted
[04:38:12] Explicit valence for atom # 3 Al, 6, is greater than permitted
[04:38:12] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:38:13] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:38:13] Explicit valence for atom # 9 Al, 6, is greater than permitted
[04:38:13] Explicit valence for atom # 5 Al, 6, is greater than permitted
[04:38:13] Explicit valence for atom # 16 Al, 6, is greater than permitted
[04:38:14] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[04:38:14] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료


In [5]:
# 셀 5
from google import genai

gemini_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_key)
print("Gemini 클라이언트 준비 완료")

Gemini 클라이언트 준비 완료


In [ ]:
%%writefile -a src/tools/agent.py


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    if len(candidates) == 1:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    # LLM에게는 name과 rationale만 보여줌 (smiles는 우리 코드가 나중에 매칭할 참조용)
    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 여러 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

이 중 이 분자 맥락에서 가장 적절한 후보를 선택하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수), "reason": "선택 이유 한 문장"}}
"""

    response = client.models.generate_content(model=model_name, contents=prompt)
    text = response.text.strip()

    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]

    try:
        result = json.loads(text)
        # idx가 유효 범위 안인지 검증
        if not isinstance(result.get('candidate_idx'), int) or not (0 <= result['candidate_idx'] < len(candidates)):
            return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
        return result
    except (json.JSONDecodeError, KeyError):
        return {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}

Appending to src/tools/agent.py


In [ ]:
multi_known_test = None
for s in data['smiles_train'][:2000]:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_known_test = s
        break

print("찾은 분자:", multi_known_test)

찾은 분자: Nc1ccc(NCCO)c([N+](=O)[O-])c1


In [ ]:
import importlib
import src.tools.agent
importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_which_candidate_to_use

decision2 = ask_llm_which_candidate_to_use(client, 'gemini-3.5-flash', multi_known_test, 'nitro_group')
print(decision2)

{'candidate_idx': 2, 'reason': 'nitrile은 니트로기의 강한 전자끌개 특성을 모방하여 주변 아민의 산화 반응성을 억제하면서도, 니트로기 특유의 대사적 유전독성 위험을 효과적으로 회피할 수 있는 최적의 대체기입니다.'}


In [ ]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    best_match = None
    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
                continue
            frag_heavy_atoms = part_mol.GetNumHeavyAtoms()
            if frag_heavy_atoms == pattern_size:
                return {"core": parts[1 - i], "target_removed": part}
            if best_match is None:
                best_match = {"core": parts[1 - i], "target_removed": part}
    return best_match


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [ ]:
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop

result_full_llm = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("상태:", result_full_llm['status'])
for h in result_full_llm['history']:
    print(h)

상태: success
{'step': 0, 'smiles': 'Nc1ccc(NCCO)c([N+](=O)[O-])c1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [10, 12]}]}
{'step': 1, 'smiles': 'N#Cc1cc(N)ccc1NCCO', 'fixed_rule': 'nitro_group', 'problem_reason': '니트로기는 체내에서 유전독성을 유발하는 반응성 대사체로 환원되기 쉬워 신약개발에서 가장 우선적으로 치환해야 하는 대표적인 독성 작용기입니다.', 'candidate_used': 'nitrile', 'candidate_reason': '니트로기와 유사한 강한 전자 끌개 성질을 유지하면서 환원성 대사 독성 위험을 제거하고, 설폰아미드 대비 입체적 장애가 적어 유효한 결합력을 유지하기에 가장 적합합니다.', 'problems': [{'rule_name': 'aniline', 'atom_indices': [2, 3, 4, 5, 6, 7, 8]}]}
{'step': 2, 'smiles': 'N#CF', 'fixed_rule': 'aniline', 'problem_reason': '유일한 치환 가능 후보', 'candidate_used': 'fluorine', 'candidate_reason': '반응성이 높은 아민기를 불소로 완전히 대체함으로써 아닐린 구조 기원의 유전독성(Ames) 및 대사 불안정성 리스크를 원천적으로 제거할 수 있기 때문입니다.', 'problems': []}


In [ ]:
core_check = find_core_and_target("N#Cc1cc(N)ccc1NCCO", "aniline")
print(core_check)

{'core': 'N#C[*:1]', 'target_removed': 'Nc1ccc(NCCO)c([*:1])c1'}


In [ ]:
from rdkit import Chem

problem_pattern = Chem.MolFromSmarts("[NH2][c]")

# 후보 1: 진짜 aniline 부분
part1 = Chem.MolFromSmiles("Nc1ccc(NCCO)c([H])c1")
print("part1 (진짜 aniline 부분) 매치:", part1.HasSubstructMatch(problem_pattern) if part1 else "파싱실패")

# 후보 2: 니트릴 조각
part2 = Chem.MolFromSmiles("N#C[H]")
print("part2 (니트릴) 매치:", part2.HasSubstructMatch(problem_pattern) if part2 else "파싱실패")
print("pattern_size:", problem_pattern.GetNumAtoms())
if part1:
    print("part1 heavy atoms:", part1.GetNumHeavyAtoms())
if part2:
    print("part2 heavy atoms:", part2.GetNumHeavyAtoms())

part1 (진짜 aniline 부분) 매치: True
part2 (니트릴) 매치: False
pattern_size: 2
part1 heavy atoms: 11
part2 heavy atoms: 2


In [ ]:
mol_check = Chem.MolFromSmiles("N#Cc1cc(N)ccc1NCCO")
fragments_check = rdMMPA.FragmentMol(mol_check, maxCuts=1, resultsAsMols=False)
for core, chain in fragments_check:
    print(f"core: {core}, chain: {chain}")

core: , chain: N#C[*:1].Nc1ccc(NCCO)c([*:1])c1
core: , chain: N#Cc1cc([*:1])ccc1NCCO.N[*:1]
core: , chain: N#Cc1cc(N)ccc1[*:1].OCCN[*:1]
core: , chain: N#Cc1cc(N)ccc1N[*:1].OCC[*:1]
core: , chain: N#Cc1cc(N)ccc1NC[*:1].OC[*:1]
core: , chain: N#Cc1cc(N)ccc1NCC[*:1].O[*:1]


In [ ]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환. 패턴 크기와 정확히 일치하는 조각만 인정.
    못 찾으면 None (차선책으로 얼버무리지 않음)."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
                continue
            frag_heavy_atoms = part_mol.GetNumHeavyAtoms()
            if frag_heavy_atoms == pattern_size:
                return {"core": parts[1 - i], "target_removed": part}
    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None):
    """진단->치환->재평가를 반복.
    llm_client가 주어지면: 어떤 문제부터 고칠지 + 어떤 후보를 쓸지 둘 다 LLM이 판단.
    없으면: 리스트 순서(known_problems[0]) + candidate_idx 고정값 사용."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패 (core/target 매칭 실패 또는 재조립 실패)", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [ ]:
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, iterative_fix_loop

# 1) 아까 문제됐던 aniline 매칭이 이번엔 어떻게 나오는지
core_check2 = find_core_and_target("N#Cc1cc(N)ccc1NCCO", "aniline")
print("aniline core_check2:", core_check2)

# 2) 전체 루프 재실행
result_full_llm2 = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("\n상태:", result_full_llm2['status'])
for h in result_full_llm2['history']:
    print(h)

aniline core_check2: None


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# 20260723 실행하지 말것
import time

result_full_llm2 = None
for attempt in range(3):
    try:
        result_full_llm2 = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
        break
    except Exception as e:
        print(f"시도 {attempt+1} 실패: {e}")
        time.sleep(10)  # 10초 대기 후 재시도

if result_full_llm2:
    print("상태:", result_full_llm2['status'])
    for h in result_full_llm2['history']:
        print(h)
else:
    print("3번 시도 모두 실패")

시도 1 실패: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 56.163202508s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'}, 'qu

In [ ]:
# 20260723 실행하지말것
result_full_llm2 = None
for attempt in range(2):
    try:
        result_full_llm2 = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-2.5-flash')
        break
    except Exception as e:
        print(f"시도 {attempt+1} 실패: {e}")
        import time
        time.sleep(5)

if result_full_llm2:
    print("상태:", result_full_llm2['status'])
    for h in result_full_llm2['history']:
        print(h)

시도 1 실패: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}
시도 2 실패: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}


In [7]:
# 셀 5까지 실행 후 바로 여기 실행
multi_known_test = None
for s in data['smiles_train'][:2000]:
    p = detect_toxicophores(s)
    known_count = sum(1 for x in p if get_replacement_candidates(x['rule_name']) is not None)
    if known_count >= 2:
        multi_known_test = s
        break

print("테스트 분자:", multi_known_test)

result_final = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("상태:", result_final['status'])
for h in result_final['history']:
    print(h)
result_final = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("상태:", result_final['status'])
for h in result_final['history']:
    print(h)

테스트 분자: Nc1ccc(NCCO)c([N+](=O)[O-])c1
상태: stuck
{'step': 0, 'smiles': 'Nc1ccc(NCCO)c([N+](=O)[O-])c1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [10, 12]}]}
{'step': 1, 'smiles': 'N#Cc1cc(N)ccc1NCCO', 'fixed_rule': 'nitro_group', 'problem_reason': '니트로기는 체내 대사 과정에서 유전독성과 변이원성을 유발하는 반응성 중간체(히드록실아민 등)로 환원될 위험이 매우 높아 최우선적으로 치환해야 합니다.', 'candidate_used': 'nitrile', 'candidate_reason': '니트릴기는 니트로기와 유사한 강한 전자끌개 성질을 유지하여 전자 밀도를 조절하면서도, 니트로기의 환원에 의한 변이원성 위험을 배제하고 대사 안정성을 크게 향상시킬 수 있는 대표적인 생체등가체입니다.', 'problems': [{'rule_name': 'aniline', 'atom_indices': [2, 3, 4, 5, 6, 7, 8]}]}
상태: stuck
{'step': 0, 'smiles': 'Nc1ccc(NCCO)c([N+](=O)[O-])c1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': 

In [8]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "michael_acceptor": {
        "problem_smarts": "C=CC(=O)",
        "candidates": [
            {"smiles": "CCC(=O)", "name": "saturated ketone",
             "rationale": "이중결합을 제거해 단백질 친전자성 부가반응(covalent binding) 위험 제거"},
        ],
    },
    "thiourea": {
        "problem_smarts": "NC(=S)N",
        "candidates": [
            {"smiles": "NC(=O)N", "name": "urea",
             "rationale": "황 원자를 산소로 대체, 유사한 형태를 유지하면서 반응성/대사 우려 감소"},
        ],
    },
    "acyl_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [9]:
import importlib
import src.tools.replacement_library
importlib.reload(src.tools.replacement_library)
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, iterative_fix_loop

# 1) 이번엔 aniline이 잘리는지 확인
core_check3 = find_core_and_target("N#Cc1cc(N)ccc1NCCO", "aniline")
print("aniline core_check3:", core_check3)

aniline core_check3: None


In [10]:
mol_check2 = Chem.MolFromSmiles("N#Cc1cc(N)ccc1NCCO")
fragments_check2 = rdMMPA.FragmentMol(mol_check2, maxCuts=1, resultsAsMols=False)
for core, chain in fragments_check2:
    print(f"core: {core}, chain: {chain}")

core: , chain: N#C[*:1].Nc1ccc(NCCO)c([*:1])c1
core: , chain: N#Cc1cc([*:1])ccc1NCCO.N[*:1]
core: , chain: N#Cc1cc(N)ccc1[*:1].OCCN[*:1]
core: , chain: N#Cc1cc(N)ccc1N[*:1].OCC[*:1]
core: , chain: N#Cc1cc(N)ccc1NC[*:1].OC[*:1]
core: , chain: N#Cc1cc(N)ccc1NCC[*:1].O[*:1]


In [11]:
# 테스트
test_frag = Chem.MolFromSmiles("N[*:1]".replace('[*:1]', 'C'))  # N[H] 대신 NC로 테스트
pattern_test = Chem.MolFromSmarts("[NH2]")
print(test_frag.HasSubstructMatch(pattern_test) if test_frag else "파싱실패")

True


In [12]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates


def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환. 패턴 크기와 정확히 일치하는 조각만 인정.
    못 찾으면 None (차선책으로 얼버무리지 않음)."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            # attachment point를 [H] 대신 C로 치환: 자유 원자가 아니라
            # "무언가에 결합되어 있던 상태"를 더 정확히 재현하기 위함
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', 'C'))
            if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
                continue
            # heavy atom 개수 비교 시, 방금 붙인 더미 탄소는 빼고 비교해야 함
            frag_heavy_atoms = part_mol.GetNumHeavyAtoms() - 1
            if frag_heavy_atoms == pattern_size:
                return {"core": parts[1 - i], "target_removed": part}
    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None):
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
        unknown_problems = [p for p in problems if p not in known_problems]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems)
            target_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')

            candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, target_rule)
            chosen_candidate_idx = candidate_decision['candidate_idx']
            candidate_reason = candidate_decision.get('reason', '')
        else:
            target_rule = known_problems[0]['rule_name']
            problem_reason = "규칙 기반(리스트 순서대로)"
            chosen_candidate_idx = candidate_idx
            candidate_reason = "규칙 기반(고정 인덱스)"

        fixed = propose_fix(current, target_rule, chosen_candidate_idx)

        if fixed is None or not fixed['is_valid']:
            return {"status": "stuck", "reason": f"'{target_rule}' 치환 실패", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history, "skipped_rules": skipped_rules}

Overwriting src/tools/molecule_editor.py


In [13]:
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import find_core_and_target, propose_fix, iterative_fix_loop

# 1) aniline이 이번엔 되는지
print("aniline:", find_core_and_target("N#Cc1cc(N)ccc1NCCO", "aniline"))

# 2) 기존에 잘 되던 것들 회귀 테스트
print("alkyl_halide:", propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print("nitro_group:", propose_fix(multi_known_test, "nitro_group", candidate_idx=0))

# 3) 전체 루프 최종 재검증
result_final2 = iterative_fix_loop(multi_known_test, max_iterations=10, llm_client=client, llm_model='gemini-3.5-flash')
print("\n최종 상태:", result_final2['status'])
for h in result_final2['history']:
    print(h)

aniline: {'core': 'N#Cc1cc([*:1])ccc1NCCO', 'target_removed': 'N[*:1]'}
alkyl_halide: {'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
nitro_group: {'new_smiles': 'Nc1ccc(NCCO)c(N)c1', 'candidate_used': 'primary amine', 'rationale': '극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함', 'is_valid': True}

최종 상태: success
{'step': 0, 'smiles': 'Nc1ccc(NCCO)c([N+](=O)[O-])c1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [0, 1, 2, 3, 4, 9, 13]}, {'rule_name': 'nitro_group', 'atom_indices': [10, 11, 12]}, {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [10, 12]}]}
{'step': 1, 'smiles': 'N#Cc1cc(N)ccc1NCCO', 'fixed_rule': 'nitro_group', 'problem_reason': '니트로기는 체내에서 유전독성(Ames mutagenicity) 및 세포독성을 유발하는 반응성 중간체로 환원될 위험이 매우 높아 신약개발에서 가장 우선적으로 제거하거나 치환해야 하는 작용기입니다.', 'candidate_used': 'nitrile', 'candidate_reason': '니트로기를 유사한 전자 끌개(EWG) 특성을 유지하면서도 돌연변이원성(Ames mutagenicity) 독성 우려가 없고 대사 안정성이 우수한 니트릴기(-

In [15]:
!git add src/tools/molecule_editor.py src/tools/replacement_library.py
!git status
!git commit -m "Fix attachment point validation: use dummy carbon instead of H (H caused false negatives due to implicit valence mismatch, e.g. NH2 vs NH3); fixes aniline rule fragmentation"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/molecule_editor.py
	modified:   src/tools/replacement_library.py

[main d447e91] Fix attachment point validation: use dummy carbon instead of H (H caused false negatives due to implicit valence mismatch, e.g. NH2 vs NH3); fixes aniline rule fragmentation
 2 files changed, 7 insertions(+), 7 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 861 bytes | 861.00 KiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   f6b54cf..d447e91  main -> main
